# Ingesta ONE-SHOT — Crisol (Raw Zone)

Objetivo: scrapear una sola vez listados de productos desde crisol.com.pe, generar snapshot crudo + deduplicado por URL normalizada en /srv/bigdata, y opcionalmente subir a HDFS: /data/raw/crisol/ingest_date=YYYY-MM-DD/. No será un proceso diario/semanal.

### Dependencias 

In [1]:
import sys,bs4, requests
print(sys.executable)
print(bs4.__version__, requests.__version__)

/home/bigdata/venvs/pyspark_env/bin/python3
4.14.2 2.32.3


### PARAMETROS

In [2]:
from datetime import datetime, date
import os, time, random

print("=== INGESTA CRISOL — ONE SHOT ===")
print("Inicio:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
RAW_DAY = date.today().isoformat()
LOCAL_DIR = "/srv/bigdata"
os.makedirs(LOCAL_DIR, exist_ok=True)
print(f"RAW_DAY : {RAW_DAY}")
print(f"LOCAL_DIR: {LOCAL_DIR}")

=== INGESTA CRISOL — ONE SHOT ===
Inicio: 2025-09-30 17:41:06
RAW_DAY : 2025-09-30
LOCAL_DIR: /srv/bigdata


## Conectividad 

In [3]:
import requests
BASE = "https://www.crisol.com.pe"    # <- base Crisol
USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
              "AppleWebKit/537.36 (KHTML, like Gecko) "
              "Chrome/120.0.0.0 Safari/537.36")
HEADERS = {"User-Agent": USER_AGENT}
REQUEST_TIMEOUT = 20

print("Chequeando HOME…", BASE, end=" ")
r = requests.get(BASE, headers=HEADERS, timeout=REQUEST_TIMEOUT, allow_redirects=True)
print("OK" if r.ok else f"FALLA ({r.status_code})")

Chequeando HOME… https://www.crisol.com.pe OK


## Utilidades

In [4]:


import re
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup

def clean_money(s):
    if not s: return None
    s = (s.replace("\u00a0"," ")
           .replace("S/.", "")
           .replace("S/", "")
           .replace("S/ ", "")
           .strip())
    digits = "".join(ch for ch in s if ch.isdigit() or ch in ".,")
    digits = digits.replace(",", "")
    try: return float(digits)
    except: return None

def fetch(url: str) -> str:
    r = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.text

# Selectores flexibles (Crisol suele ser ecommerce con grillas tipo Magento/VTEX)
SELECTORS = {
    "product_card": [
        "li.product", "li.product-item", "div.product-item",
        "div.product.product-item", "div.product-grid-item",
        "ol.products li", "ul.products li", "div.products div.product"
    ],
    "title": [
        "a.product-item-link", "h2.product.name a", "h3.product-title a", "a.product.name",
        "a.product-title", "h2.product-title a"
    ],
    "price_current": [
        "span.price-final_price span.price", "span.special-price span.price",
        "span.price-wrapper span.price", "span.price", "span.product-price"
    ],
    "price_old": [
        "span.old-price span.price", "del span.price",
        "span.price-wrapper .old-price span.price", "del.price", "span.list-price"
    ],
    "link": [
        "a.product-item-link", "h2.product.name a", "a.product.name", "h3.product-title a",
        "a.product-title"
    ],
    "next_page": [
        "a[title='Siguiente']", "a.next", "a[rel='next']",
        "li.pages-item-next a", "a.action.next", "a.pagination-next"
    ],
}

def first_text(el, candidates):
    for sel in candidates:
        f = el.select_one(sel)
        if f and f.get_text(strip=True):
            return f.get_text(strip=True)
    return ""

def first_attr(el, candidates, attr="href"):
    for sel in candidates:
        f = el.select_one(sel)
        if f and f.get(attr):
            return f.get(attr)
    return ""

def url_to_catname(u: str) -> str:
    p = urlparse(u).path.strip("/").replace(".html","")
    parts = [x for x in p.split("/") if x]
    return " / ".join([x.replace("-", " ").title() for x in parts[:4]]) or "General"

## Taxonomia

In [5]:
## Taxonomia

import unicodedata, json

taxonomy = {
  "taxonomy_version": "2025-09-screenshots",
  "top": ["Los más vendidos","Novedades","Ficción","No ficción","Infantil","Juvenil","Smart Play","Tecnología"],
  "Los más vendidos": {"type": "collection", "children": ["Los más vendidos del año","Los 100 más vendidos del mes"]},
  "Novedades": {"type": "collection", "children": ["Nuevos lanzamientos","Preventa"]},
  "Ficción": {"groups": {
      "Resaltados (colecciones)": ["Voces Peruanas","Vargas Llosa","Premios","Juego de tronos","Star wars"],
      "Narrativa": ["Literatura clásica","Literatura Latinoamericana","Literatura Peruana","Literatura Universal","Literatura en otros idiomas","Literatura de viajes"],
      "Novelas": ["Ciencia ficción","Fantasía","Novela histórica","Novela negra","Romántica y erótica","Terror"],
      "Lírica y drama": ["Poesía","Teatro"],
      "Mangas": ["Adulto","Seinen","Shojo","Shonen"],
      "Cómics y novela gráfica": ["Ciencia ficción, fantasía y horror","Clásicos de la literatura","Cómic de héroes","Cómic infantil","Erótico","Cómic Europeo","Humor","Cómics en inglés","Novela gráfica","Cómic Americano","Libros de ilustración","Cómic Peruano"]
  }},
  "No ficción": {"groups": {
      "Resaltados (colecciones)": ["Los más vendidos","Actualidad Peruana","Artículos del mes","Rosa María Cifuentes","Gift cards"],
      "Libros de no ficción": ["Ciencias","Humanidades","Ciencias sociales","Desarrollo personal y salud","Empresa"],
      "Artes e ilustrados / Escolar / Técnicos": ["Artes e ilustrados","Escolar y plan lector","Gastronomía","Libro técnico","Deportes"],
      "Consulta y otros": ["Obras de consulta","Turismo","Miscelánea","Enseñanza de idiomas"]
  }, "deep": { "Humanidades": ["Biografías","Comunicación","Educación","Educación en el Perú","Ensayo","Estudios literarios y lingüística","Estudios literarios y lingüística del Perú","Filosofía","Geografía","Historia del Perú","Mitología","Psicoanálisis","Psicología","Religión"]}},
  "Infantil": {"groups": {
      "De 0 a 2 años": ["Cuentos imprescindibles","Hábitos y emociones","Libros con solapa","Primeros conocimientos","Tus personajes favoritos"],
      "De 3 a 5 años": ["Álbumes ilustrados","Cuentos clásicos","Libros para colorear","Prelectura y escritura","Primeros conocimientos","Tus personajes favoritos","Valores"],
      "De 6 a 8 años": ["Literatura de 6 a 8 años","Álbumes ilustrados","Ciencia","Cultura y sociales","Libros con valores","Libros para colorear","Libros que te conectan a la lectura","Manualidades","Primeros idiomas","Tus personajes favoritos"],
      "De 9 a 12 años": ["Literatura de 9 a 12","Ciencia","Cocina y manualidades","Cultura y sociales"]
  }},
  "Juvenil": {"groups": {
      "Wattpad": ["Wattpad"],
      "Romance spicy": ["Romance"],
      "Comics y mangas": ["Shojo","Shonen","Comics de héroes","One piece"],
      "Literatura juvenil": ["Terror y misterio","Fantasía","Ciencia ficción","Youtubers","Drama","Clásicos"],
      "Fantasía": ["Fantasía"]
  }},
  "Smart Play": {"groups": {
      "Resaltados (colecciones)": ["Los más vendidos","Novedades","Mundo Harry Potter","Productos Steam","Artículos del mes","Gift cards"],
      "Smart play": ["Rompecabezas","Juguetes educativos","Juegos de mesa","Vinilos"],
      "Complementos": ["Merchandising","Agendas y libretas","Útiles de escritorio"]
  }},
  "Tecnología": {"groups": {
      "Audífonos": ["Audífonos deportivos/open ear","Audífonos headphones/on ear","Audífonos in ear","Audífonos inalámbricos"],
      "Cargadores": ["Cargadores para auto","Cargadores de carga rápida","Cargadores portátiles","Cables de carga"],
      "Parlantes & Altavoces": ["Parlantes","Parlantes mini"]
  }}
}

In [6]:
def slugify(name: str) -> str:
    s = unicodedata.normalize("NFKD", name).encode("ascii","ignore").decode("ascii")
    s = s.lower()
    s = s.replace("&"," y ").replace("/", " ").replace(".", " ")
    s = re.sub(r"\s+y\s+"," y ", s)
    s = re.sub(r"[^a-z0-9\s-]","", s)
    s = re.sub(r"\s+","-", s).strip("-")
    return s

def flatten_taxonomy(obj):
    """Aplana el JSON en una lista de 'rutas' candidatas (en texto)"""
    paths = set()
    def add(label, prefix=""):
        p = "/".join(x for x in [prefix, slugify(label)] if x)
        paths.add(p)
        return p
    # top
    for t in taxonomy.get("top", []):
        add(t)
    # niveles conocidos
    for key, val in taxonomy.items():
        if isinstance(val, dict):
            if "children" in val:
                base = add(key)
                for c in val["children"]:
                    add(c, base)
            if "groups" in val:
                base = add(key)
                for grp, items in val["groups"].items():
                    g = add(grp, base)
                    for it in items:
                        add(it, g)
            if "deep" in val:
                base = add(key)
                for grp, items in val["deep"].items():
                    g = add(grp, base)
                    for it in items:
                        add(it, g)
    return sorted(paths)

candidates = flatten_taxonomy(taxonomy)

def check_url_variants(path_slug):
    """Prueba URL con y sin .html (algunos sitios usan ambas formas)."""
    variants = [
        f"{BASE}/{path_slug}.html",
        f"{BASE}/{path_slug}",
    ]
    for u in variants:
        try:
            r = requests.get(u, headers=HEADERS, timeout=REQUEST_TIMEOUT, allow_redirects=True)
            if r.status_code == 200:
                return u
        except Exception:
            pass
    return None

CATEGORIES = []
for p in candidates:
    u = check_url_variants(p)
    if u: CATEGORIES.append(u)

print("=== Categorías validadas (Crisol) ===")
print("Total:", len(CATEGORIES))
for u in CATEGORIES[:12]:
    print(" -", u)

assert CATEGORIES, "No se validó ninguna categoría de Crisol; revisa estructura/slug/robots."

=== Categorías validadas (Crisol) ===
Total: 67
 - https://www.crisol.com.pe/ficcion
 - https://www.crisol.com.pe/ficcion/comics-y-novela-grafica
 - https://www.crisol.com.pe/ficcion/comics-y-novela-grafica/ciencia-ficcion-fantasia-y-horror
 - https://www.crisol.com.pe/ficcion/comics-y-novela-grafica/clasicos-de-la-literatura
 - https://www.crisol.com.pe/ficcion/comics-y-novela-grafica/comic-de-heroes
 - https://www.crisol.com.pe/ficcion/comics-y-novela-grafica/comic-infantil
 - https://www.crisol.com.pe/ficcion/comics-y-novela-grafica/erotico
 - https://www.crisol.com.pe/ficcion/comics-y-novela-grafica/humor
 - https://www.crisol.com.pe/ficcion/comics-y-novela-grafica/libros-de-ilustracion
 - https://www.crisol.com.pe/ficcion/comics-y-novela-grafica/novela-grafica
 - https://www.crisol.com.pe/ficcion/lirica-y-drama
 - https://www.crisol.com.pe/ficcion/lirica-y-drama/poesia


## Parser de listados

In [8]:
## Parser de listados


def parse_listing(html: str, base_url: str):
    soup = BeautifulSoup(html, "lxml")

    # Tarjetas
    cards = []
    for sel in SELECTORS["product_card"]:
        cards = soup.select(sel)
        if cards: break

    rows = []
    for card in cards:
        title = first_text(card, SELECTORS["title"])
        link  = first_attr(card, SELECTORS["link"], "href")
        pcur  = first_text(card, SELECTORS["price_current"])
        pold  = first_text(card, SELECTORS["price_old"])
        rows.append({
            "titulo": title,
            "autor": "",  # opcional (otra pasada)
            "precio_actual": clean_money(pcur),
            "precio_anterior": clean_money(pold),
            "url_producto": urljoin(base_url, link) if link else "",
        })

    # Siguiente página
    next_url = ""
    for sel in SELECTORS["next_page"]:
        a = soup.select_one(sel)
        if a and a.get("href"):
            next_url = urljoin(base_url, a.get("href"))
            break

    return rows, next_url

## Crawl multi-categoría

In [9]:
## Crawl multi-categoría

from datetime import datetime
import pandas as pd

SLEEP_RANGE = (1.5, 3.0)   # un poco más largo si corres en paralelo con SBS
MAX_PAGES   = 30           # tope de seguridad

def crawl_category(cat_url, max_pages=MAX_PAGES, sleep_range=SLEEP_RANGE):
    url   = cat_url
    page  = 1
    data  = []
    seen  = set()
    while url and page <= max_pages:
        if url in seen: break
        seen.add(url)
        try:
            html = fetch(url)
        except Exception as e:
            print("[error]", e, "->", url); break

        items, next_url = parse_listing(html, url)
        for it in items:
            it["categoria"] = url_to_catname(cat_url)
        data.extend(items)

        print(f"[info] {url_to_catname(cat_url)} | página {page}: {len(items)} items")
        url  = next_url
        page += 1
        time.sleep(random.uniform(*sleep_range))
    return data

# === correr ===
all_rows = []
t0 = datetime.now()
print("\n=== INICIANDO CRAWL (CRISOL) ===")
for i, cat in enumerate(CATEGORIES, 1):
    t_cat = datetime.now()
    print(f"[{i}/{len(CATEGORIES)}] {cat}")
    data_cat = crawl_category(cat)
    all_rows.extend(data_cat)
    print(f" -> items categ.: {len(data_cat)} | acumulado: {len(all_rows)} | t: {datetime.now()-t_cat}")
    print("-"*60)
print("=== FIN CRAWL ===")
print("Total filas crudas:", len(all_rows))
print("Tiempo total:", datetime.now()-t0)

df = pd.DataFrame(
    all_rows,
    columns=["titulo","autor","precio_actual","precio_anterior","url_producto","categoria"]
)
df.insert(0, "fecha_scraping", datetime.now().strftime("%Y-%m-%d"))
print("Preview:"); df.head(3)


=== INICIANDO CRAWL (CRISOL) ===
[1/67] https://www.crisol.com.pe/ficcion
[info] Ficcion | página 1: 15 items
[info] Ficcion | página 2: 15 items
[info] Ficcion | página 3: 15 items
[info] Ficcion | página 4: 13 items
[info] Ficcion | página 5: 15 items
[info] Ficcion | página 6: 15 items
[info] Ficcion | página 7: 14 items
[info] Ficcion | página 8: 15 items
[info] Ficcion | página 9: 15 items
[info] Ficcion | página 10: 15 items
[info] Ficcion | página 11: 15 items
[info] Ficcion | página 12: 15 items
[info] Ficcion | página 13: 15 items
[info] Ficcion | página 14: 15 items
[info] Ficcion | página 15: 15 items
[info] Ficcion | página 16: 15 items
[info] Ficcion | página 17: 15 items
[info] Ficcion | página 18: 15 items
[info] Ficcion | página 19: 15 items
[info] Ficcion | página 20: 15 items
[info] Ficcion | página 21: 15 items
[info] Ficcion | página 22: 15 items
[info] Ficcion | página 23: 15 items
[info] Ficcion | página 24: 13 items
[info] Ficcion | página 25: 15 items
[info] Fi

,fecha_scraping,titulo,autor,precio_actual,precio_anterior,url_producto,categoria
0,2025-09-30,El amor en los tiempos del cóle...,,57.85,89.0,https://www.crisol.com.pe/libro-el-amor-en-los...,Ficcion
1,2025-09-30,El psicoanalista,,38.35,59.0,https://www.crisol.com.pe/libro-el-psicoanalis...,Ficcion
2,2025-09-30,El castillo,,38.35,59.0,https://www.crisol.com.pe/libro-el-castillo-97...,Ficcion


## Guardar SNAPSHOT

In [10]:
## Guardar SNAPSHOT

TS = datetime.now().strftime("%Y%m%d_%H%M%S")
SNAPSHOT_OUT = os.path.join(LOCAL_DIR, f"crisol_precios_allcats_{TS}.csv")
df.to_csv(SNAPSHOT_OUT, index=False, encoding="utf-8-sig")
size_mb = os.path.getsize(SNAPSHOT_OUT)/1024/1024
print(f"[OK] Guardado crudo: {SNAPSHOT_OUT} ({size_mb:.2f} MB)")

[OK] Guardado crudo: /srv/bigdata/crisol_precios_allcats_20250930_182420.csv (1.87 MB)


## Verificaciones

In [11]:
## Verificaciones

print("Nulos por columna:"); print(df.isna().sum().sort_values(ascending=False))
print("\nDuplicados por url_producto:", df.duplicated("url_producto").sum())
print("\nTop categorías:"); print(df["categoria"].value_counts().head(10))

Nulos por columna:
precio_anterior    69
fecha_scraping      0
titulo              0
autor               0
precio_actual       0
url_producto        0
categoria           0
dtype: int64

Duplicados por url_producto: 4505

Top categorías:
categoria
Smart Play / Complementos / Utiles De Escritorio    449
Ficcion / Comics Y Novela Grafica                   448
Infantil                                            448
Ficcion / Novelas                                   448
No Ficcion / Humanidades                            447
Ficcion / Narrativa                                 447
No Ficcion                                          446
Juvenil                                             446
Ficcion                                             445
Juvenil / Literatura Juvenil                        445
Name: count, dtype: int64


## Deduplicado por URL normalizada

In [12]:
## Deduplicado por URL normalizada

url = df["url_producto"].astype(str).str.strip()
url_norm = (url.str.lower()
              .str.replace(r"#.*$", "", regex=True)
              .str.replace(r"\?.*$", "", regex=True)
              .str.rstrip("/"))
df["url_norm"] = url_norm

print("Productos (url_norm) únicos antes de dedup:", df["url_norm"].nunique())

catmap = (df.assign(categoria=df["categoria"].astype(str).str.strip())
            .groupby("url_norm")["categoria"]
            .apply(lambda s: " | ".join(sorted(set([c for c in s if c]))))
            .reset_index(name="categorias_join"))

df["_score"] = df["precio_actual"].notna().astype(int)
keep = (df.sort_values(["url_norm","fecha_scraping","_score"], ascending=[True, False, False])
          .drop_duplicates("url_norm", keep="first")
          .drop(columns=["_score"])
          .copy())

keep = keep.merge(catmap, on="url_norm", how="left")
cols = ["fecha_scraping","titulo","autor","precio_actual","precio_anterior",
        "url_producto","categoria","categorias_join","url_norm"]
keep = keep[cols]

DEDUP_OUT = os.path.join(LOCAL_DIR, "crisol_snapshot_dedup.csv")
keep.to_csv(DEDUP_OUT, index=False, encoding="utf-8-sig")

print("Filas originales:", len(df))
print("Productos únicos:", keep["url_norm"].nunique())
print("[OK] Guardado dedup:", DEDUP_OUT)## Subir HDFS

Productos (url_norm) únicos antes de dedup: 8210
Filas originales: 12715
Productos únicos: 8210
[OK] Guardado dedup: /srv/bigdata/crisol_snapshot_dedup.csv


## Subir HDFS

In [13]:
## Subir HDFS

import subprocess
HDFS_DIR = f"/data/raw/crisol/ingest_date={RAW_DAY}/"
print("Subiendo a HDFS:", HDFS_DIR)
subprocess.run(["hdfs","dfs","-mkdir","-p", HDFS_DIR], check=False)
subprocess.run(["hdfs","dfs","-put","-f", SNAPSHOT_OUT, HDFS_DIR], check=True)
subprocess.run(["hdfs","dfs","-put","-f", DEDUP_OUT, HDFS_DIR], check=True)
subprocess.run(["hdfs","dfs","-touchz", f"{HDFS_DIR}_SUCCESS"], check=False)
print("[HDFS] OK ->", HDFS_DIR)

Subiendo a HDFS: /data/raw/crisol/ingest_date=2025-09-30/
[HDFS] OK -> /data/raw/crisol/ingest_date=2025-09-30/
